# Selezione Dinamica della Dimensione del Campione - NSynth con Rete Neurale

Esperimento **esplorativo** (fuori dalle ipotesi convesse della tesi, Sez. 6.5): gli stessi algoritmi dei listati B.1-B.4 applicati a una rete neurale a due strati (non convessa).

**Nota importante**: le funzioni per-esempio `loss_i`/`grad_i`/`hessvec_i` dei listati sono qui rimpiazzate dalle versioni batch matriciali `loss_batch`/`grad_batch`/`hess_batch`, matematicamente equivalenti (la cella 3b lo verifica numericamente) ma calcolate senza loop Python, per evitare il crash su Colab. La logica degli algoritmi (Wolfe, CCV, CG, ortante) è identica.

In [ ]:
#@title 0. Dipendenze e import
%pip install -q librosa scikit-learn autograd

import os, json, time, tarfile, urllib.request
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import librosa
from sklearn.preprocessing import RobustScaler

# autograd per l'Hessiano-vettore su batch (una sola chiamata)
from autograd import hessian_vector_product
import autograd.numpy as anp

print("numpy", np.__version__)
print("librosa", librosa.__version__)


In [ ]:
#@title 1. Scarica ed estrae NSynth (validation e test)
BASE = "http://download.magenta.tensorflow.org/datasets/nsynth/"

def scarica_estrai(split):
    if os.path.isdir(f"nsynth-{split}") and os.path.isdir(f"nsynth-{split}/audio"):
        print(f"OK: split {split} già presente")
        return
    fname = f"nsynth-{split}.jsonwav.tar.gz"
    if not os.path.exists(fname):
        print(f"Scaricamento {fname} (~{'1 GB' if split=='valid' else '350 MB'}) ...")
        urllib.request.urlretrieve(BASE + fname, fname)
    print(f"Estrazione {fname} ...")
    with tarfile.open(fname, "r:gz") as tar:
        tar.extractall()
    print(f"OK: split {split} pronto")

for s in ("valid", "test"):
    scarica_estrai(s)

import glob
print("clip valid =", len(glob.glob("nsynth-valid/audio/*.wav")))
print("clip test  =", len(glob.glob("nsynth-test/audio/*.wav")))


In [ ]:
#@title 2. Estrazione features audio (pipeline ottimizzata)
SR, N_MELS, HOP, N_FFT = 16000, 96, 256, 1024
N_MFCC    = 40
N_CQT     = 84
CACHE     = True

import concurrent.futures as cf

def estrai_features(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    eps = 1e-10
    y_h, y_p = librosa.effects.hpss(y)

    S    = librosa.feature.melspectrogram(y=y_h, sr=SR, n_mels=N_MELS,
                                          n_fft=N_FFT, hop_length=HOP)
    logS = librosa.power_to_db(S, ref=np.max)
    dS   = librosa.feature.delta(logS)
    d2S  = librosa.feature.delta(logS, order=2)
    f = [logS.mean(1), logS.std(1), dS.std(1), d2S.std(1)]

    mfcc  = librosa.feature.mfcc(y=y_h, sr=SR, n_mfcc=N_MFCC,
                                  n_fft=N_FFT, hop_length=HOP)
    dmfcc = librosa.feature.delta(mfcc)
    f += [mfcc.mean(1), mfcc.std(1), dmfcc.mean(1), dmfcc.std(1)]

    C_cqt = np.abs(librosa.cqt(y_h, sr=SR, hop_length=HOP, n_bins=N_CQT))
    logC  = librosa.amplitude_to_db(C_cqt, ref=np.max)
    f += [logC.mean(1), logC.std(1)]

    tn = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=SR)
    f += [tn.mean(1), tn.std(1)]

    ct = librosa.feature.spectral_contrast(y=y_h, sr=SR, n_fft=N_FFT, hop_length=HOP)
    ch = librosa.feature.chroma_stft(y=y_h, sr=SR, n_fft=N_FFT, hop_length=HOP)
    f += [ct.mean(1), ct.std(1), ch.mean(1), ch.std(1)]

    for v in (librosa.feature.spectral_centroid(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.spectral_bandwidth(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.spectral_rolloff(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP),
              librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP)):
        f += [v.mean(1), v.std(1)]

    ons = librosa.onset.onset_strength(y=y, sr=SR, hop_length=HOP)
    Eh, Ep = float(np.sum(y_h ** 2)), float(np.sum(y_p ** 2))
    rms  = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)[0]
    tt   = np.arange(len(rms))
    tc   = float(np.sum(tt * rms) / (np.sum(rms) + eps)) / max(len(rms) - 1, 1)
    f.append(np.array([ons.mean(), ons.std(), ons.max(),
                       np.log((Eh + eps) / (Ep + eps)),
                       np.log((Ep + eps) / (Eh + Ep + eps)),
                       float(rms.max() / (rms.mean() + eps)),
                       tc, float(np.log1p(np.sum(y ** 2)))]))
    return np.concatenate(f).astype(np.float32)

def _estrai_uno(arg):
    jsondir, name, fam_idx = arg
    fv = estrai_features(os.path.join(jsondir, "audio", name + ".wav"))
    return fv, fam_idx

def costruisci_dati(split):
    jsondir = f"nsynth-{split}"
    cache_f = os.path.join(jsondir, "features_opt_net.npz")
    with open(os.path.join(jsondir, "examples.json"), encoding="utf-8") as fjs:
        notes = json.load(fjs)
    fams  = sorted({md["instrument_family_str"] for md in notes.values()})
    fam2i = {f: i for i, f in enumerate(fams)}
    items = [(n, md) for n, md in notes.items()
             if os.path.exists(os.path.join(jsondir, "audio", n + ".wav"))]
    if CACHE and os.path.exists(cache_f):
        z = np.load(cache_f)
        if list(z["names"]) == [n for n, _ in items]:
            print(f"[{split}] feature caricate da cache: {z['X'].shape}")
            return z["X"], z["y"], fams
    args = [(jsondir, n, fam2i[md["instrument_family_str"]]) for n, md in items]
    t0 = time.time()
    nw = os.cpu_count() or 1
    if nw > 1:
        with cf.ProcessPoolExecutor(max_workers=nw) as ex:
            out = list(ex.map(_estrai_uno, args, chunksize=16))
    else:
        out = [_estrai_uno(a) for a in args]
    X = np.asarray([o[0] for o in out], np.float32)
    y = np.asarray([o[1] for o in out], np.int64)
    print(f"[{split}] {len(items)} clip -> {X.shape[1]} feature "
          f"in {(time.time() - t0) / 60:.1f} min")
    if CACHE:
        np.savez_compressed(cache_f, X=X, y=y,
                            names=np.array([n for n, _ in items]))
    return X, y, fams

Xtr, Ytr, FAMILIES = costruisci_dati("valid")
Xte, Yte, _        = costruisci_dati("test")
print("Xtr", Xtr.shape, " Xte", Xte.shape)
print("famiglie (10):", FAMILIES)

scaler = RobustScaler(quantile_range=(5.0, 95.0))
Xtr = scaler.fit_transform(Xtr).astype(np.float32)
Xte = scaler.transform(Xte).astype(np.float32)
Xtr = np.clip(Xtr, -10, 10); Xte = np.clip(Xte, -10, 10)
Xtr = np.nan_to_num(Xtr);    Xte = np.nan_to_num(Xte)
print("Dimensione feature:", Xtr.shape[1])


In [ ]:
#@title 3. Definizione del problema: rete neurale a due strati (NON lineare)
# ============================================================================
# Architettura: input_dim -> 256 (tanh) -> 128 (tanh) -> 10 (softmax).
# I parametri sono appiattiti in un unico vettore w.
# ATTENZIONE: le funzioni loss_i/grad_i/hessvec_i PER-ESEMPIO dei listati
# B.1-B.4 sono qui sostituite dalle versioni BATCH MATRICIALI equivalenti
# (loss_batch/grad_batch/hess_batch): stessi valori, ma calcolati senza
# loop Python su N=12678, per evitare il crash su Colab.
# La regolarizzazione L2 (LAM) è applicata solo ai pesi (non ai bias).
# ============================================================================
N = Xtr.shape[0]
input_dim = Xtr.shape[1]
h1, h2 = 256, 128
out_dim = 10
LAM = 1e-4          # regolarizzazione L2 (sui pesi)
R = 0.1             # rapporto per sottocampionamento dell'Hessiano (Newton)

def param_slices(input_dim, h1, h2, out_dim):
    idx = 0
    W1_s = slice(idx, idx + input_dim*h1); idx += input_dim*h1
    b1_s = slice(idx, idx + h1);           idx += h1
    W2_s = slice(idx, idx + h1*h2);        idx += h1*h2
    b2_s = slice(idx, idx + h2);           idx += h2
    W3_s = slice(idx, idx + h2*out_dim);   idx += h2*out_dim
    b3_s = slice(idx, idx + out_dim);      idx += out_dim
    return W1_s, b1_s, W2_s, b2_s, W3_s, b3_s
W1_s, b1_s, W2_s, b2_s, W3_s, b3_s = param_slices(input_dim, h1, h2, out_dim)

rng_init = np.random.RandomState(42)
W1_init = rng_init.randn(input_dim, h1) * np.sqrt(2.0 / (input_dim + h1))
b1_init = np.zeros(h1)
W2_init = rng_init.randn(h1, h2) * np.sqrt(2.0 / (h1 + h2))
b2_init = np.zeros(h2)
W3_init = rng_init.randn(h2, out_dim) * np.sqrt(2.0 / (h2 + out_dim))
b3_init = np.zeros(out_dim)
w0 = np.concatenate([W1_init.ravel(), b1_init, W2_init.ravel(),
                     b2_init, W3_init.ravel(), b3_init]).astype(float)
print("Parametri totali:", w0.size)

# ---------- forward pass vettorizzato ----------
def forward(w, Xb):
    W1 = w[W1_s].reshape(input_dim, h1); b1 = w[b1_s]
    W2 = w[W2_s].reshape(h1, h2);        b2 = w[b2_s]
    W3 = w[W3_s].reshape(h2, out_dim);   b3 = w[b3_s]
    z1 = Xb @ W1 + b1; a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2; a2 = np.tanh(z2)
    logits = a2 @ W3 + b3
    return logits, a1, a2

# ---------- loss media su batch (equivalente a mean_i loss_i) ----------
def loss_batch(w, indices):
    Xb = Xtr[indices]; Yb = Ytr[indices]; m = len(indices)
    logits, _, _ = forward(w, Xb)
    logits = logits - logits.max(axis=1, keepdims=True)
    P = np.exp(logits); P /= P.sum(axis=1, keepdims=True)
    ce = -np.log(P[np.arange(m), Yb] + 1e-15)
    W1 = w[W1_s].reshape(input_dim, h1)
    W2 = w[W2_s].reshape(h1, h2)
    W3 = w[W3_s].reshape(h2, out_dim)
    reg = 0.5 * LAM * (np.sum(W1**2) + np.sum(W2**2) + np.sum(W3**2))
    return float(np.mean(ce) + reg)

# ---------- gradiente medio su batch (equivalente a mean_i grad_i) ----------
def grad_batch(w, indices):
    Xb = Xtr[indices]; Yb = Ytr[indices]; m = len(indices)
    logits, a1, a2 = forward(w, Xb)
    logits = logits - logits.max(axis=1, keepdims=True)
    P = np.exp(logits); P /= P.sum(axis=1, keepdims=True)
    Yoh = np.zeros_like(P); Yoh[np.arange(m), Yb] = 1.0
    dlog = (P - Yoh) / m
    W1 = w[W1_s].reshape(input_dim, h1); b1 = w[b1_s]
    W2 = w[W2_s].reshape(h1, h2);        b2 = w[b2_s]
    W3 = w[W3_s].reshape(h2, out_dim);   b3 = w[b3_s]
    dW3 = a2.T @ dlog; db3 = dlog.sum(axis=0)
    da2 = dlog @ W3.T; dz2 = da2 * (1 - a2**2)
    dW2 = a1.T @ dz2; db2 = dz2.sum(axis=0)
    da1 = dz2 @ W2.T; dz1 = da1 * (1 - a1**2)
    dW1 = Xb.T @ dz1; db1 = dz1.sum(axis=0)
    dW1 += LAM * W1; dW2 += LAM * W2; dW3 += LAM * W3
    return np.concatenate([dW1.ravel(), db1, dW2.ravel(), db2,
                           dW3.ravel(), db3]).astype(float)
# ---------- varianza per-esempio esatta per la CCV (fedele al listato B.1) ---
def per_example_sq_norms(w, indices, chunk=32):
    """Somma di ||g_i||^2 sui gradienti per-esempio, a blocchi (memoria OK)."""
    m = len(indices); s2 = 0.0
    for s in range(0, m, chunk):
        idx = indices[s:s+chunk]
        Xb = Xtr[idx]; Yb = Ytr[idx]; mc = len(idx)
        logits, a1, a2 = forward(w, Xb)
        logits = logits - logits.max(axis=1, keepdims=True)
        P = np.exp(logits); P /= P.sum(axis=1, keepdims=True)
        Yoh = np.zeros_like(P); Yoh[np.arange(mc), Yb] = 1.0
        dlog = P - Yoh
        W1 = w[W1_s].reshape(input_dim, h1); b1 = w[b1_s]
        W2 = w[W2_s].reshape(h1, h2);        b2 = w[b2_s]
        W3 = w[W3_s].reshape(h2, out_dim);   b3 = w[b3_s]
        dW3 = np.einsum('bi,bj->bij', a2, dlog).reshape(mc, -1)
        db3 = dlog
        da2 = dlog @ W3.T; dz2 = da2 * (1 - a2**2)
        dW2 = np.einsum('bi,bj->bij', a1, dz2).reshape(mc, -1)
        db2 = dz2
        da1 = dz2 @ W2.T; dz1 = da1 * (1 - a1**2)
        dW1 = np.einsum('bi,bj->bij', Xb, dz1).reshape(mc, -1)
        db1 = dz1
        reg_vec = np.concatenate([(LAM*W1).ravel(), np.zeros(h1),
                                  (LAM*W2).ravel(), np.zeros(h2),
                                  (LAM*W3).ravel(), np.zeros(out_dim)])
        G = np.concatenate([dW1, db1, dW2, db2, dW3, db3], axis=1) + reg_vec[None, :]
        s2 += float(np.sum(G*G))
    return s2

def ccv_stats(w, indices, chunk=32):
    """Varianza per-esempio del listato B.1:
    V_norm1 = (sum_i ||g_i||^2 - n*||gbar||^2) / (n-1)
            == np.var(grads, axis=0, ddof=1).sum()"""
    g = grad_batch(w, indices)
    n = len(indices)
    if n <= 1:
        return 0.0, float(np.dot(g, g))
    s2 = per_example_sq_norms(w, indices, chunk)
    V_norm1 = (s2 - n * float(np.dot(g, g))) / (n - 1)
    return V_norm1, float(np.dot(g, g))

# ---------- gradiente/loss completi (un solo passaggio matriciale) ----------
def grad_full(w):
    return grad_batch(w, np.arange(N))

def loss_full(w):
    return loss_batch(w, np.arange(N))

# ---------- Hessiano-vettore su batch (una sola chiamata autograd) ----------
def _fwd_anp(wf, Xb):
    W1 = anp.reshape(wf[W1_s], (input_dim, h1)); b1 = wf[b1_s]
    W2 = anp.reshape(wf[W2_s], (h1, h2));        b2 = wf[b2_s]
    W3 = anp.reshape(wf[W3_s], (h2, out_dim));   b3 = wf[b3_s]
    z1 = anp.dot(Xb, W1) + b1; a1 = anp.tanh(z1)
    z2 = anp.dot(a1, W2) + b2; a2 = anp.tanh(z2)
    return anp.dot(a2, W3) + b3

def hess_batch(w, indices, v):
    Xb = Xtr[indices]; Yb = Ytr[indices]
    def f(wf):
        logits = _fwd_anp(wf, Xb)
        logits = logits - anp.max(logits, axis=1, keepdims=True)
        p = anp.exp(logits); p = p / anp.sum(p, axis=1, keepdims=True)
        ce = -anp.log(p[anp.arange(len(Xb)), Yb] + 1e-15)
        W1 = anp.reshape(wf[W1_s], (input_dim, h1))
        W2 = anp.reshape(wf[W2_s], (h1, h2))
        W3 = anp.reshape(wf[W3_s], (h2, out_dim))
        reg = 0.5 * LAM * (anp.sum(W1**2) + anp.sum(W2**2) + anp.sum(W3**2))
        return anp.mean(ce) + reg
    hv = hessian_vector_product(f, argnum=0)
    return hv(w, v)

# ---------- accuratezza sul test ----------
def acc_test(w):
    logits, _, _ = forward(w, Xte)
    pred = np.argmax(logits, axis=1)
    return float(np.mean(pred == Yte))

print("Accuratezza iniziale (w0):", acc_test(w0)*100, "%")


In [ ]:
#@title 3b. VALIDAZIONE MATEMATICA (vettorizzato == per-esempio)
# Verifica su un piccolo batch che le funzioni batch coincidano con le
# versioni per-esempio dei listati (stesso seed, stessi valori). Se stampa
# OK, l'accuracy NON cambia rispetto alla versione per-esempio.
from autograd import grad as _agrad

def _loss_i(w, i):
    Xb = Xtr[[i]]; Yb = Ytr[[i]]
    def f(wf):
        logits = _fwd_anp(wf, Xb)
        logits = logits - anp.max(logits, axis=1, keepdims=True)
        p = anp.exp(logits); p = p / anp.sum(p, axis=1, keepdims=True)
        ce = -anp.log(p[anp.arange(1), Yb] + 1e-15)
        W1 = anp.reshape(wf[W1_s], (input_dim, h1))
        W2 = anp.reshape(wf[W2_s], (h1, h2))
        W3 = anp.reshape(wf[W3_s], (h2, out_dim))
        reg = 0.5 * LAM * (anp.sum(W1**2) + anp.sum(W2**2) + anp.sum(W3**2))
        return anp.mean(ce) + reg
    return float(f(w))

def _grad_i(w, i):
    Xb = Xtr[[i]]; Yb = Ytr[[i]]
    def f(wf):
        logits = _fwd_anp(wf, Xb)
        logits = logits - anp.max(logits, axis=1, keepdims=True)
        p = anp.exp(logits); p = p / anp.sum(p, axis=1, keepdims=True)
        ce = -anp.log(p[anp.arange(1), Yb] + 1e-15)
        W1 = anp.reshape(wf[W1_s], (input_dim, h1))
        W2 = anp.reshape(wf[W2_s], (h1, h2))
        W3 = anp.reshape(wf[W3_s], (h2, out_dim))
        reg = 0.5 * LAM * (anp.sum(W1**2) + anp.sum(W2**2) + anp.sum(W3**2))
        return anp.mean(ce) + reg
    return _agrad(f)(w)

def _hessvec_i(w, i, v):
    Xb = Xtr[[i]]; Yb = Ytr[[i]]
    def f(wf):
        logits = _fwd_anp(wf, Xb)
        logits = logits - anp.max(logits, axis=1, keepdims=True)
        p = anp.exp(logits); p = p / anp.sum(p, axis=1, keepdims=True)
        ce = -anp.log(p[anp.arange(1), Yb] + 1e-15)
        W1 = anp.reshape(wf[W1_s], (input_dim, h1))
        W2 = anp.reshape(wf[W2_s], (h1, h2))
        W3 = anp.reshape(wf[W3_s], (h2, out_dim))
        reg = 0.5 * LAM * (anp.sum(W1**2) + anp.sum(W2**2) + anp.sum(W3**2))
        return anp.mean(ce) + reg
    return hessian_vector_product(f, argnum=0)(w, v)

rng_v = np.random.RandomState(0)
idx_v = rng_v.choice(N, size=8, replace=False)
w_v = w0.copy() + 0.01 * rng_v.randn(w0.size)
ok_v = True

e1 = np.abs(grad_batch(w_v, idx_v) - np.mean([_grad_i(w_v, i) for i in idx_v], axis=0)).max()
e2 = abs(loss_batch(w_v, idx_v) - np.mean([_loss_i(w_v, i) for i in idx_v]))
VV, _ = ccv_stats(w_v, idx_v)
grads_v = np.array([_grad_i(w_v, i) for i in idx_v])
e3 = abs(VV - np.sum(np.var(grads_v, axis=0, ddof=1)))
vv = rng_v.randn(w0.size)
e4 = np.abs(hess_batch(w_v, idx_v, vv) - np.mean([_hessvec_i(w_v, i, vv) for i in idx_v], axis=0)).max()

print(f"grad_batch vs mean grad_i   : max|err| = {e1:.2e}")
print(f"loss_batch vs mean loss_i   : err      = {e2:.2e}")
print(f"ccv_stats vs np.var         : err      = {e3:.2e}")
print(f"hess_batch vs mean hessvec  : max|err| = {e4:.2e}")
ok_v = (e1 < 1e-8) and (e2 < 1e-10) and (e3 < 1e-8) and (e4 < 1e-6)
print("VALIDAZIONE:", "OK" if ok_v else "FAIL")


#### Listati B.1-B.4 (logica identica, calcolo batch matriciale)

In [ ]:
#@title 4. I quattro algoritmi (LISTATI B.1-B.4) - LOGICA IDENTICA, CALCOLO BATCH
# ============================================================================
# La logica matematica (Wolfe, CCV, CG, proiezione ortante) è IDENTICA ai
# listati B.1-B.4. Cambia solo il calcolo di gradiente/loss/Hessiano, ora
# matriciale su batch (grad_batch/loss_batch/hess_batch) al posto dei loop
# per-esempio. La varianza della CCV è quella per-esempio esatta.
# ============================================================================
import numpy as np

# ---- Listato B.1 - Dynamic GD (CCV + line search di Wolfe) ----
def dynamic_gd(w0, theta, max_iter, alpha, batch0):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history = [w.copy().tolist()]
    batch_sizes = [n]
    for k in range(max_iter):
        indices = np.random.choice(N, size=n, replace=False)
        g = grad_batch(w, indices)
        V_norm1, gg = ccv_stats(w, indices)
        if gg > 1e-16:
            if V_norm1 / n > theta ** 2 * gg:
                n_new = int(np.ceil(V_norm1 / (theta ** 2 * gg))) + 1
                n = min(n_new, N)
        c1, c2 = 1e-4, 0.9
        step = alpha
        J_curr = loss_batch(w, indices)
        g_norm2 = np.dot(g, g)
        d = -g
        gd = -g_norm2
        if g_norm2 > 1e-16:
            for _ in range(30):
                w_new = w + step * d
                if loss_batch(w_new, indices) <= J_curr + c1 * step * gd:
                    g_new = grad_batch(w_new, indices)
                    if np.dot(g_new, d) >= c2 * gd:
                        break
                step *= 0.5
            else:
                step = 0.0
        w = w + step * d
        if np.linalg.norm(grad_full(w)) < 1e-6:
            history.append(w.copy().tolist())
            batch_sizes.append(n)
            break
        history.append(w.copy().tolist())
        batch_sizes.append(n)
    return history, batch_sizes

# ---- Listato B.2 - Newton-CG ----
def cg(A, b, gamma, maxcg):
    x = np.zeros_like(b)
    r = b - A(x)
    p = r.copy()
    rr = np.dot(r, r)
    for _ in range(maxcg):
        Ap = A(p)
        pHp = np.dot(p, Ap)
        if pHp <= 1e-14:
            break
        alpha = rr / pHp
        x = x + alpha * p
        r_new = r - alpha * Ap
        rr_new = np.dot(r_new, r_new)
        if rr_new <= gamma * np.dot(x, x) + 1e-16:
            return x
        beta = rr_new / rr
        p = r_new + beta * p
        r = r_new
        rr = rr_new
    return x

def newton_cg(w0, theta, max_iter, alpha, batch0, R, maxcg):
    # Nota: gamma=0 (la varianza per-esempio degli Hv è proibitiva su D~234k).
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history, batch_sizes = [w.copy().tolist()], [n]
    for k in range(max_iter):
        indices_S = np.random.choice(N, size=n, replace=False)
        g = grad_batch(w, indices_S)
        n_h = min(max(1, int(round(R * n))), N)
        indices_H = np.random.choice(indices_S, size=n_h, replace=False)
        Hv = lambda v: hess_batch(w, indices_H, v)
        p0 = -g
        p0_norm2 = np.dot(p0, p0)
        gamma = 0.0
        d = cg(Hv, -g, gamma, maxcg)
        c1, c2 = 1e-4, 0.9
        step, J_w = alpha, loss_batch(w, indices_S)
        gd = np.dot(g, d)
        if gd >= 0:
            d = -g
            gd = -np.dot(g, g)
        for _ in range(30):
            w_new = w + step * d
            if loss_batch(w_new, indices_S) <= J_w + c1 * step * gd:
                g_new = grad_batch(w_new, indices_S)
                if np.dot(g_new, d) >= c2 * gd:
                    break
            step *= 0.5
        else:
            step = 0.0
        w = w + step * d
        history.append(w.copy().tolist())
        batch_sizes.append(n)
        indices_new = np.random.choice(N, size=n, replace=False)
        g_new = grad_batch(w, indices_new)
        V_norm1, gg_new = ccv_stats(w, indices_new)
        if gg_new > 1e-16 and V_norm1 / n > theta ** 2 * gg_new:
            n = min(int(np.ceil(V_norm1 / (theta ** 2 * gg_new))) + 1, N)
        if np.linalg.norm(grad_full(w)) < 1e-6:
            break
    return history, batch_sizes
# ---- Listato B.3 - Newton-CG L1 (subgradiente + active set + ortante) ----
def _subgrad_batch(v, indices):
    gJ = grad_batch(v, indices)
    g = np.zeros_like(gJ)
    g[v > 0] = gJ[v > 0] + nu
    g[v < 0] = gJ[v < 0] - nu
    z = (v == 0)
    g[z & (gJ < -nu)] = gJ[z & (gJ < -nu)] + nu
    g[z & (gJ > nu)] = gJ[z & (gJ > nu)] - nu
    g[z & (gJ >= -nu) & (gJ <= nu)] = 0.0
    return g

def _project_orthant(v, z):
    res = v.copy()
    m = (z != 0) & (np.sign(res) != z)
    res[m] = 0.0
    return res

def newton_l1(w0, theta, max_iter, alpha, batch0, nu_, sigma, maxcg, eta=0.5):
    global nu
    nu = nu_
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history = [w.copy().tolist()]
    batch_sizes = [n]

    def F_batch(v, indices):
        return loss_batch(v, indices) + nu * np.sum(np.abs(v))

    for k in range(max_iter):
        indices_S = np.random.choice(N, size=n, replace=False)
        g_batch = grad_batch(w, indices_S)
        z = np.where(w > 0, 1,
                     np.where(w < 0, -1,
                              np.where(g_batch < -nu, 1,
                                       np.where(g_batch > nu, -1, 0))))
        sg = _subgrad_batch(w, indices_S)
        sgn = np.linalg.norm(sg)
        if sgn < 1e-10:
            history.append(w.copy().tolist())
            batch_sizes.append(n)
            break
        n_h = max(1, int(round(R * n)))
        n_h = min(n_h, N)
        indices_H = np.random.choice(indices_S, size=n_h, replace=False)
        free = (z != 0)
        d = np.zeros_like(w)
        if np.any(free):
            g_free = sg[free]
            def Hv(v_full):
                return hess_batch(w, indices_H, v_full)
            def Hv_free(v_free):
                v_full = np.zeros_like(w)
                v_full[free] = v_free
                return Hv(v_full)[free]
            tol_cg = eta * np.linalg.norm(g_free)
            d_free = np.zeros(np.sum(free))
            r = -g_free.copy()
            p = r.copy()
            rr = np.dot(r, r)
            for _ in range(maxcg):
                Hp = Hv_free(p)
                pHp = np.dot(p, Hp)
                if pHp <= 1e-14:
                    if np.linalg.norm(d_free) < 1e-14:
                        d_free = -g_free.copy()
                    break
                alpha_cg = rr / pHp
                d_free = d_free + alpha_cg * p
                r_new = r - alpha_cg * Hp
                rr_new = np.dot(r_new, r_new)
                if np.sqrt(rr_new) <= tol_cg:
                    r = r_new
                    rr = rr_new
                    break
                beta = rr_new / rr
                p = r_new + beta * p
                r = r_new
                rr = rr_new
            d[free] = d_free
        step = alpha
        F_w = F_batch(w, indices_S)
        sg_d = np.dot(sg, d)
        if sg_d >= 0:
            d = -sg
            sg_d = -np.dot(sg, sg)
        w_new = w.copy()
        for _ in range(20):
            w_trial = _project_orthant(w + step * d, z)
            if F_batch(w_trial, indices_S) <= F_w + sigma * step * sg_d:
                w_new = w_trial
                break
            step *= 0.5
            if step < 1e-12:
                w_new = w.copy()
                break
        w = w_new
        indices_new = np.random.choice(N, size=n, replace=False)
        g_new = grad_batch(w, indices_new)
        V_norm1, gg_new = ccv_stats(w, indices_new)
        if gg_new > 1e-16:
            if V_norm1 / n > theta ** 2 * gg_new:
                n_new = int(np.ceil(V_norm1 / (theta ** 2 * gg_new))) + 1
                n = min(n_new, N)
        history.append(w.copy().tolist())
        batch_sizes.append(n)
        if np.linalg.norm(grad_full(w)) < 1e-6:
            break
    return history, batch_sizes
# ---- Listato B.4 - BB-CCV (Barzilai-Borwein + CCV + Armijo) ----
def bb_dynamic_gd(w0, theta, max_iter, alpha, batch0):
    w = np.array(w0, dtype=float)
    n = max(batch0, 2)
    history = [w.copy().tolist()]
    batch_sizes = [n]
    w_prev = w.copy()
    g_prev = None
    for k in range(max_iter):
        indices = np.random.choice(N, size=n, replace=False)
        g = grad_batch(w, indices)
        if k > 0 and g_prev is not None:
            s = w - w_prev
            y = g - g_prev
            sy = np.dot(s, y)
            if abs(sy) > 1e-14:
                step_bb = np.dot(s, s) / sy
                step = np.clip(step_bb, alpha / 20.0, alpha * 5.0)
            else:
                step = alpha
        else:
            step = alpha
        w_prev = w.copy()
        g_prev = g.copy()
        c1 = 1e-4
        J_curr = loss_batch(w, indices)
        g_norm2 = np.dot(g, g)
        if g_norm2 > 1e-16:
            for _ in range(30):
                w_new = w - step * g
                if loss_batch(w_new, indices) <= J_curr - c1 * step * g_norm2:
                    break
                step *= 0.5
            else:
                step = 0.0
        w = w - step * g
        V_norm1, gg = ccv_stats(w, indices)
        if gg > 1e-16:
            if V_norm1 / n > theta ** 2 * gg:
                n_new = int(np.ceil(V_norm1 / (theta ** 2 * gg))) + 1
                n = min(n_new, N)
        history.append(w.copy().tolist())
        batch_sizes.append(n)
        if np.linalg.norm(grad_full(w)) < 1e-6:
            break
    return history, batch_sizes


In [ ]:
#@title 5. TEST VELOCE (solo Dynamic GD, poche iterazioni)
# Serve a confermare che tutto gira prima di lanciare l'esecuzione completa.
import time

print("Avvio Dynamic GD (test veloce, 40 iterazioni, batch 128)...")
np.random.seed(42)
t0 = time.time()
hist_t, bs_t = dynamic_gd(w0, 0.3, 40, 1.0, 128)
dt = time.time() - t0
wf_t = np.array(hist_t[-1])
print(f"Completato in {dt:.1f} s  |  loss {loss_full(w0):.4f} -> "
      f"{loss_full(wf_t):.4f}  |  acc test = {acc_test(wf_t)*100:.2f}%  "
      f"|  batch_fin = {max(bs_t)}")


In [ ]:
#@title 6. ESECUZIONE DEI QUATTRO METODI (listati B.1-B.4)
# Tempi stimati (Colab CPU): Dynamic GD/BB-CCV ~15-45 min, Newton-CG dipende
# da MAXCG (qui 30). Se vuoi una prova rapida imposta MAX_ITER=100.
SEED, THETA, ALPHA, BATCH0, MAX_ITER = 42, 0.3, 1.0, 128, 400
NU, SIGMA, MAXCG = 5e-4, 1e-4, 30

methods = [
    ("Dynamic GD",   lambda: dynamic_gd(w0, THETA, MAX_ITER, ALPHA, BATCH0)),
    ("Newton-CG",    lambda: newton_cg(w0, THETA, MAX_ITER, ALPHA, BATCH0, R, MAXCG)),
    ("Newton-CG L1", lambda: newton_l1(w0, THETA, MAX_ITER, ALPHA, BATCH0, NU, SIGMA, MAXCG)),
    ("BB-CCV",       lambda: bb_dynamic_gd(w0, THETA, MAX_ITER, ALPHA, BATCH0)),
]

def subgrad_full_l1(w):
    gJ = grad_full(w)
    g = np.zeros_like(gJ)
    g[w > 0] = gJ[w > 0] + NU
    g[w < 0] = gJ[w < 0] - NU
    z = (w == 0)
    g[z & (gJ < -NU)] = gJ[z & (gJ < -NU)] + NU
    g[z & (gJ > NU)] = gJ[z & (gJ > NU)] - NU
    g[z & (gJ >= -NU) & (gJ <= NU)] = 0.0
    return g

results, accs, batches, pesi = {}, {}, {}, {}
for name, run in methods:
    np.random.seed(SEED)
    t0 = time.time()
    history, batch_sizes = run()
    dt = time.time() - t0
    wf = np.array(history[-1])
    pesi[name] = wf.copy()
    if name == "Newton-CG L1":
        gnorm = float(np.linalg.norm(subgrad_full_l1(wf)))
        nnz = int(np.sum(wf != 0))
    else:
        gnorm = float(np.linalg.norm(grad_full(wf)))
        nnz = None
    acc = acc_test(wf)
    results[name] = dict(iter=len(history)-1, acc=acc, loss=loss_full(wf),
                         gnorm=gnorm, batch=max(batch_sizes), batch0=batch_sizes[0],
                         time=dt, nnz=nnz)
    accs[name] = [acc_test(np.array(w)) for w in history]
    batches[name] = batch_sizes
    print(f"{name:14s} iter={results[name]['iter']:3d}  "
          f"acc={results[name]['acc']*100:5.2f}%  "
          f"||grad||={gnorm:.1e}  tempo={dt:5.1f}s"
          + (f"  nnz={nnz}" if nnz is not None else ""))

print("\nEsecuzione completata. Salvataggio risultati...")
import json as _json
_json.dump(results, open("results_net.json", "w"), indent=2)
print("Salvato: results_net.json")


In [ ]:
#@title 7. Figura 1 - accuratezza sul test vs iterazioni
COLORS = {"Dynamic GD": "#1f77b4", "Newton-CG": "#d62728",
          "Newton-CG L1": "#2ca02c", "BB-CCV": "#9467bd"}
ks = np.arange(0, MAX_ITER + 1)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
for name, _ in methods:
    a = np.asarray(accs[name]) * 100
    ax.plot(ks[:len(a)], a, lw=1.8, label=name, color=COLORS[name])
ax.axhline(20.8, color="gray", ls="--", lw=1.2, label="Classe maggioritaria")
ax.set_xlabel(r"Iterazione $k$")
ax.set_ylabel(r"Accuratezza sul test (\%)")
ax.set_title("NSynth (rete neurale): accuratezza di test vs iterazioni")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("nsynth_accuracy_net.png", dpi=150)
fig.savefig("nsynth_accuracy_net.pdf")
plt.show()
print("Salvate: nsynth_accuracy_net.png / .pdf")


In [ ]:
#@title 8. Figura 2 - dinamica del batch n_k vs k
fig, ax = plt.subplots(figsize=(6.4, 4.0))
for name, _ in methods:
    b = np.asarray(batches[name])
    ax.step(np.arange(len(b)), b, where="mid", lw=1.6, label=name,
            color=COLORS[name])
ax.set_xlabel(r"Iterazione $k$")
ax.set_ylabel(r"Dimensione del batch $n_k$")
ax.set_title("NSynth (rete neurale): dinamica del batch (CCV)")
ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("nsynth_batch_net.png", dpi=150)
fig.savefig("nsynth_batch_net.pdf")
plt.show()
print("Salvate: nsynth_batch_net.png / .pdf")


In [ ]:
#@title 9. Tabella dei risultati (pronta per il LaTeX)
print("=== Tabella risultati (rete neurale) ===")
print(r"Metodo & Acc. test & $\|\nabla J(w)\|_2$ & Batch finale & Tempo (s) & Coeff. non nulli \\")
for name, _ in methods:
    r = results[name]
    nnz = f"{r['nnz']}/{w0.size}" if r["nnz"] is not None else "---"
    print(f"{name} & {r['acc']*100:.1f}\\% & ${r['gnorm']:.1e}$ & "
          f"{r['batch']} & {r['time']:.1f} & {nnz} \\\\")

print("\n=== Trade-off sparsità/accuratezza (Newton-CG L1) ===")
print(r"$\nu$ & Acc. test & Coeff. non nulli & Sparsità \\")
for nu_ in (1e-4, 1e-3, 5e-3):
    np.random.seed(SEED)
    h, bs = newton_l1(w0, THETA, MAX_ITER, ALPHA, BATCH0, nu_, SIGMA, MAXCG)
    wf = np.array(h[-1])
    a = acc_test(wf)
    nz = int(np.sum(wf != 0))
    print(f"${nu_:.0e}$ & {a*100:.1f}\\% & {nz}/{w0.size} & {100*(1-nz/w0.size):.0f}\\% \\\\")


In [ ]:
#@title 10. (Opzionale) Scarica figure e risultati
try:
    from google.colab import files
    for f in ("nsynth_accuracy_net.png", "nsynth_accuracy_net.pdf",
              "nsynth_batch_net.png", "nsynth_batch_net.pdf", "results_net.json"):
        files.download(f)
except ImportError:
    print("Non sei in Colab: i file sono nella cartella corrente.")
    print(os.listdir("."))


In [ ]:
#@title 11. (Facoltativo) Ascolta un esempio e confronta le predizioni
from IPython.display import Audio
import json, random

names_test = list(json.load(open("nsynth-test/examples.json")).keys())
i = random.randint(0, len(names_test)-1)
print(f"Esempio {i}: {names_test[i]}  |  Reale: {FAMILIES[Yte[i]]}")
for name, w in pesi.items():
    logits, _, _ = forward(w, Xte[[i]])
    pred = FAMILIES[np.argmax(logits[0])]
    print(f"  {name:14s}: {pred}  {'OK' if pred == FAMILIES[Yte[i]] else 'NO'}")
display(Audio(f"nsynth-test/audio/{names_test[i]}.wav"))
